# Stage 9 — model-ready forensic features

Prepare a reproducible subscriber-hour modeling table from canonical CDRs. The target is whether a SIM is fraud-labeled. Each row represents one SIM at the end of an observed hour; rolling features use events up to and including that hour. Identifiers and ground-truth-derived columns are excluded from model inputs.

This provides both a static SIM profile and a history-aware hourly view. Because the source covers one day, history means same-day history only.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED=Path('../outputs/processed/all_naive_12pct_200sims')
OUTPUT=Path('../outputs/model_ready/all_naive_12pct_200sims')
OUTPUT.mkdir(parents=True,exist_ok=True)
cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string','src_imei':'string','dst_imei':'string'},parse_dates=['timestamp'])
fraud=pd.read_csv(PROCESSED/'fraudulent_users.csv',dtype={'src_number':'string'})['src_number'].dropna().tolist(); fraud=set(fraud)
print('CDR records:',len(cdr),'fraud labels:',len(fraud),'time range:',cdr.timestamp.min(),cdr.timestamp.max())
assert cdr.record_id.is_unique
assert cdr.timestamp.notna().all()

In [ ]:
# Canonical events become source-SIM activity rows; this matches the fraud-label role in this scenario.
voice={'MOC','MTC','I-MOC','I-MTC'}; outgoing={'MOC','I-MOC','MO-SMS'}; incoming={'MTC','I-MTC'}; international={'I-MOC','I-MTC'}
e=cdr.copy(); e['sim']=e.src_number; e['contact']=e.dst_number; e['hour_ts']=e.timestamp.dt.floor('h'); e['known_fraud']=e.sim.isin(fraud)
e['is_voice']=e.cdr_type.isin(voice); e['is_outgoing']=e.cdr_type.isin(outgoing); e['is_incoming']=e.cdr_type.isin(incoming); e['is_international']=e.cdr_type.isin(international); e['is_night']=e.timestamp.dt.hour.between(0,5); e['is_short_voice']=e.is_voice & e.duration.le(10)
print('Source SIMs observed:',e.sim.nunique(),'fraud SIMs observed:',e.loc[e.known_fraud,'sim'].nunique())

In [ ]:
# Build SIM-hour aggregates, including quiet hours after each SIM's first observed event.
hourly=e.groupby(['sim','hour_ts'],sort=True).agg(events=('record_id','nunique'),calls=('is_voice','sum'),outgoing=('is_outgoing','sum'),incoming=('is_incoming','sum'),international=('is_international','sum'),short_voice=('is_short_voice','sum'),duration_sum=('duration','sum'),duration_mean=('duration','mean'),unique_contacts=('contact','nunique'),unique_imeis=('src_imei','nunique'),unique_cells=('current_src_cell_id','nunique'),night_events=('is_night','sum'),fraud=('known_fraud','max')).reset_index()
# User identity is retained only as a grouping key and removed from X later.
rows=[]
for sim,g in hourly.groupby('sim',sort=False):
    g=g.sort_values('hour_ts').set_index('hour_ts')
    full_index=pd.date_range(g.index.min().floor('h'),g.index.max().floor('h'),freq='h')
    g=g.reindex(full_index)
    g['sim']=sim; g['fraud']=g['fraud'].ffill().bfill().astype(int)
    count_cols=['events','calls','outgoing','incoming','international','short_voice','duration_sum','night_events']
    g[count_cols]=g[count_cols].fillna(0)
    for c in ['duration_mean','unique_contacts','unique_imeis','unique_cells']: g[c]=g[c].fillna(0)
    g['hour_ts']=g.index; rows.append(g.reset_index(drop=True))
hourly=pd.concat(rows,ignore_index=True)
hourly=hourly.sort_values(['sim','hour_ts'])
print('SIM-hour rows:',len(hourly),'positive rows:',int(hourly.fraud.sum()))

In [ ]:
# History features are causal: rolling windows include current hour; lag columns use only prior hours.
for col in ['events','calls','outgoing','incoming','international','short_voice','duration_sum','unique_contacts','night_events']:
    grp=hourly.groupby('sim',sort=False)[col]
    hourly[f'{col}_lag1']=grp.shift(1).fillna(0)
    hourly[f'{col}_prev3h']=hourly.groupby('sim',sort=False)[col].transform(lambda s:s.shift(1).rolling(3,min_periods=1).sum()).fillna(0)
    hourly[f'{col}_cum']=hourly.groupby('sim',sort=False)[col].cumsum()
hourly['hour_of_day']=hourly.hour_ts.dt.hour
hourly['elapsed_hours']=hourly.groupby('sim').cumcount()
hourly['international_ratio']=hourly.international/hourly.events.replace(0,np.nan)
hourly['outgoing_ratio']=hourly.outgoing/hourly.events.replace(0,np.nan)
hourly['short_voice_ratio']=hourly.short_voice/hourly.calls.replace(0,np.nan)
hourly['activity_change_vs_prev']=hourly.events-hourly.events_lag1

# Static profile is written separately for the static model; never join full-day values into hourly rows.
static=e.groupby('sim').agg(static_events=('record_id','nunique'),static_calls=('is_voice','sum'),static_outgoing=('is_outgoing','sum'),static_incoming=('is_incoming','sum'),static_international=('is_international','sum'),static_duration=('duration','sum'),static_contacts=('contact','nunique'),static_active_hours=('hour_ts','nunique')).reset_index()
static['static_fraud']=static.sim.isin(fraud).astype(int)
static.to_csv(OUTPUT/'static_sim_features.csv',index=False)
features=hourly.copy()
features.to_csv(OUTPUT/'history_aware_sim_hour_features.csv',index=False)
print('saved static and SIM-hour tables')

In [ ]:
# Define safe model inputs and run leakage/quality checks.
exclude={'sim','hour_ts','fraud','known_fraud','static_fraud','src_number','dst_number','record_id','event_key','src_known_fraud','dst_known_fraud','involves_known_fraud'}
feature_columns=[c for c in features.columns if c not in exclude and pd.api.types.is_numeric_dtype(features[c])]
X=features[feature_columns].replace([np.inf,-np.inf],np.nan)
y=features['fraud'].astype(int)
assert set(y.unique())=={0,1},f'Expected both labels, got {set(y.unique())}'
assert not any(c in feature_columns for c in ['fraud','known_fraud','static_fraud'])
assert features.groupby('sim').fraud.nunique().max()==1
quality=pd.DataFrame({'dtype':X.dtypes.astype(str),'missing_rate':X.isna().mean(),'unique_values':X.nunique(dropna=True)})
quality.to_csv(OUTPUT/'feature_quality_report.csv')
pd.Series(feature_columns,name='feature').to_csv(OUTPUT/'model_feature_columns.csv',index=False)
print('positive SIMs:',features.loc[features.fraud.eq(1),'sim'].nunique(),'negative SIMs:',features.loc[features.fraud.eq(0),'sim'].nunique())
print('model features:',len(feature_columns)); print(quality.sort_values('missing_rate',ascending=False).head(15))

## Modeling guidance

Use `static_sim_features.csv` for the static model and `history_aware_sim_hour_features.csv` for the temporal model. Split by SIM so a subscriber never appears in both train and test sets. For early-detection evaluation, evaluate cumulative predictions by hour and report SIM-level detection yield. Do not include `sim`, `hour_ts`, or label columns in model inputs. `hour_of_day` is retained as a behavioral feature; test an ablation without it if simulator timing may encode scenario artifacts.